# 気象庁 過去の気象データ 一括ダウンロード

[観測所ファインダー](https://awg-yk.github.io/weather-station-finder/) で選んだ
**地点リスト**を使って、気象庁の過去の気象データをまとめて取得するノートブックです。

## 使い方
1. 下のセルを実行（▶ボタン）すると**入力フォーム**が表示されます
2. フォームの **1 → 2 → 3** の順に選びます:
   - **1. 地点リスト**: ファインダーの「Colab用にコピー」を押し、貼り付け欄に貼る
   - **2. データの種類 ＆ 観測項目** を選ぶ
   - **3. 期間の種類 ＆ 開始〜終了** を選ぶ
3. 「ダウンロード開始」を押すと、ボタン下の**進捗バー**に状況が出て、最後にZIPが手元に落ちてきます

- 既定の期間は「1976年1月1日 〜 昨日」（細かく設定しない場合の目安）
- **気象庁の1回あたりデータ量上限を超えないよう、期間を自動で分割**して取得します
- まとめ方は「地点ごと／期間ごと／自動」。**期間ごとは複数地点を1リクエストにまとめて取得するため速い**（自動はリクエストが少ない方を選択）
- 途中で失敗しても、再実行すれば取得済みはスキップして続きから取得します
- 気象庁サーバーに負荷をかけないよう、1件ずつ間隔（3秒）を空けて取得します


In [ ]:
# ============================================================
# 気象庁 過去の気象データ 一括ダウンロード（Google Colab・フォーム版）
#
# 観測所ファインダー( https://awg-yk.github.io/weather-station-finder/ )で
# 出力したCSVを入力に、指定した種類・項目・期間のデータをまとめて取得します。
#
# 使い方:
#   1. このセルを実行（▶）するとフォームが表示されます
#   2. 地点リストを渡す（どちらか）:
#        (A) ファインダーの「Colab用にコピー」を押し、フォームの「または貼付」欄に貼り付け
#        (B) 「選択結果をCSVでダウンロード」で保存したCSVを「CSVを選択」でアップロード
#   3. データの種類・観測項目・期間などをプルダウンで選択
#   4. 「ダウンロード開始」ボタンを押すと取得が進み、最後にZIPが手元に落ちてきます
#
# 特徴:
#   - すべてプルダウン等のフォームで選択（横スクロールで隠れる問題を解消）
#   - 「連続した期間」と「特定の期間を複数年分」の2モードに対応
#   - 取得後、地点ごとに1ファイルへ自動結合してファイル数を削減
#   - 地点番号(prec_no/block_no)はCSVから直接取得。取得済みファイルはスキップ再開
#   - 気象庁サーバーに負荷をかけないよう、1件ずつスリープを挟んで取得します
# ============================================================

!pip install -q requests ipywidgets

import calendar
import csv as csv_module
import io
import json
import re
import shutil
import time
from dataclasses import asdict, dataclass, field
from datetime import date, timedelta
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from urllib.parse import parse_qs, urlparse

import requests
import ipywidgets as widgets
from IPython.display import display
from google.colab import files

ROOT_URL = "https://www.data.jma.go.jp/risk/obsdl/index.php"
SHOW_URL = "https://www.data.jma.go.jp/risk/obsdl/show/table"
STATIONS_JSON_URL = "https://raw.githubusercontent.com/awg-yk/weather-station-finder/main/data/stations.json"

MAX_RETRIES = 3

# 集計期間: (コード, 表示名, 連続モードでの分割単位)
PERIOD_OPTIONS: List[Tuple[str, str, str]] = [
    ("9", "時別値",     "month"),
    ("1", "日別値",     "year"),
    ("2", "半旬別値",   "year"),
    ("4", "旬別値",     "year"),
    ("5", "月別値",     "year"),
    ("6", "3か月別値",  "year"),
]

# 時別値(9)用の観測項目: (コード, 表示名, カテゴリ)
ELEMENTS_HOURLY: List[Tuple[str, str, str]] = [
    ("201", "気温", "気温"),
    ("101", "降水量（前1時間）", "降水量"),
    ("301", "風向・風速", "風"),
    ("401", "日照時間（前1時間）", "日照時間"),
    ("610", "全天日射量（前1時間）", "日照時間"),
    ("501", "積雪の深さ", "積雪"),
    ("503", "降雪の深さ（前1時間）", "積雪"),
    ("605", "相対湿度", "湿度"),
    ("604", "蒸気圧", "湿度"),
    ("612", "露点温度", "湿度"),
    ("601", "現地気圧", "湿度"),
    ("602", "海面気圧", "湿度"),
    ("607", "雲量", ""),
    ("703", "天気", ""),
    ("704", "視程", ""),
]

# 日別/半旬/旬/月/3か月 共通の観測項目: (コード, 表示名, 対応期間集合, カテゴリ)
ELEMENTS_OTHER: List[Tuple[str, str, set, str]] = [
    ("201", "平均気温",            {"1", "2", "4", "5", "6"}, "気温"),
    ("202", "最高気温",            {"1", "2", "4", "5", "6"}, "気温"),
    ("203", "最低気温",            {"1", "2", "4", "5", "6"}, "気温"),
    ("204", "日最高気温の平均",    {"2", "4", "5", "6"}, "気温"),
    ("206", "日最低気温の平均",    {"2", "4", "5", "6"}, "気温"),
    ("205", "日最高気温の最低",    {"2", "4", "5", "6"}, "気温"),
    ("207", "日最低気温の最高",    {"2", "4", "5", "6"}, "気温"),
    ("101", "降水量の合計",        {"1", "2", "4", "5", "6"}, "降水量"),
    ("102", "日降水量の最大",      {"2", "4", "5", "6"}, "降水量"),
    ("401", "日照時間",            {"1", "2", "4", "5", "6"}, "日照時間"),
    ("610", "合計全天日射量",      {"1", "2", "4", "5", "6"}, "日照時間"),
    ("501", "最深積雪",            {"1", "2", "4", "5", "6"}, "積雪"),
    ("503", "降雪量の合計",        {"1", "2", "4", "5", "6"}, "積雪"),
    ("504", "降雪量日合計の最大",  {"2", "4", "5", "6"}, "積雪"),
    ("301", "平均風速",            {"1", "2", "4", "5", "6"}, "風"),
    ("302", "最大風速（風向）",    {"1", "2", "4", "5", "6"}, "風"),
    ("304", "最大瞬間風速（風向）", {"1", "2", "4", "5", "6"}, "風"),
    ("305", "最多風向",            {"1", "2", "4", "5", "6"}, "風"),
    ("605", "平均相対湿度",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("606", "最小相対湿度",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("604", "平均蒸気圧",          {"1", "2", "4", "5", "6"}, "湿度"),
    ("601", "平均現地気圧",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("602", "平均海面気圧",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("603", "最低海面気圧",        {"1", "2", "4", "5", "6"}, "湿度"),
    ("607", "平均雲量",            {"1", "2", "4", "5", "6"}, ""),
    ("701", "天気概況（昼）",      {"1"}, ""),
    ("702", "天気概況（夜）",      {"1"}, ""),
]

ID_COLUMN_CANDIDATES = ["観測所ID", "地点コード", "地点番号"]


@dataclass
class Station:
    station_id: str
    name: str
    prec_no: str
    block_no: str
    station_type: str
    elements: set
    status: str

    def station_num(self) -> str:
        # obsdlの地点番号は block_no の桁数で接頭辞が決まる:
        #   5桁(WMO 47xxx = 気象官署) → "s"+block_no、4桁以下(アメダス) → "a"+ゼロ埋め4桁。
        # 「種別」ラベルでは判定しない（例: つくば/館野 47646 は種別アメダスでも obsdl では s47646）。
        if len(self.block_no) >= 5:
            return "s" + self.block_no
        return "a" + self.block_no.zfill(4)


@dataclass
class WeatherDataPayload:
    stationNumList: List[str] = field(default_factory=list)
    aggrgPeriod: int = 1
    elementNumList: List[List[str]] = field(default_factory=list)
    interAnnualType: int = 1
    ymdList: List[str] = field(default_factory=list)  # [y1, y2, m1, m2, d1, d2]
    optionNumList: List[Any] = field(default_factory=list)
    downloadFlag: str = "true"
    rmkFlag: int = 1
    disconnectFlag: int = 1
    youbiFlag: int = 0
    fukenFlag: int = 0
    kijiFlag: int = 0
    huukouFlag: int = 0
    csvFlag: int = 1
    jikantaiFlag: int = 0
    jikantaiList: List[Any] = field(default_factory=list)
    ymdLiteral: int = 1

    def to_post_data(self) -> dict:
        data = {}
        for key, value in asdict(self).items():
            data[key] = json.dumps(value) if isinstance(value, list) else value
        return data


# ------------------------------------------------------------
# 入力CSVの読み込み
# ------------------------------------------------------------
def parse_prec_block_from_url(url: str) -> Tuple[Optional[str], Optional[str]]:
    if not url:
        return None, None
    try:
        q = parse_qs(urlparse(url).query)
        return q.get("prec_no", [None])[0], q.get("block_no", [None])[0]
    except Exception:
        return None, None


def parse_elements_cell(cell: str) -> set:
    if not cell:
        return set()
    return {p.strip() for p in re.split(r"[\/／,、]", cell) if p.strip()}


def load_master_fallback() -> Dict[str, dict]:
    cache = Path("stations_master.json")
    if not cache.exists():
        resp = requests.get(STATIONS_JSON_URL, timeout=30)
        resp.raise_for_status()
        cache.write_bytes(resp.content)
    raw = json.loads(cache.read_text(encoding="utf-8"))
    return {str(s["id"]): s for s in raw["stations"]}


def find_id_column(fieldnames: List[str]) -> Optional[str]:
    for cand in ID_COLUMN_CANDIDATES:
        if cand in fieldnames:
            return cand
    return None


def read_stations_from_text(text: str) -> List[Station]:
    text = text.lstrip("﻿")  # 貼り付け時に残るBOMを除去（列名の頭に付くと列検出に失敗するため）
    reader = csv_module.DictReader(io.StringIO(text))
    fields = reader.fieldnames or []
    id_col = find_id_column(fields)
    if id_col is None:
        raise ValueError(f"CSVに地点IDの列（{'／'.join(ID_COLUMN_CANDIDATES)}）が見つかりません。列: {fields}")
    rows = list(reader)

    has_url = "気象庁ページURL" in fields
    master = None if has_url else load_master_fallback()

    out: List[Station] = []
    for row in rows:
        sid = (row.get(id_col) or "").strip()
        if not sid:
            continue
        name = (row.get("地点名") or "").strip()
        stype = (row.get("種別") or "").strip()
        status = (row.get("状態") or "").strip()
        elems = parse_elements_cell(row.get("観測要素", ""))

        prec = block = None
        if has_url:
            prec, block = parse_prec_block_from_url(row.get("気象庁ページURL", ""))
        if (not prec or not block) and master is not None:
            m = master.get(sid)
            if m:
                prec, block = m.get("precNo"), m.get("blockNo")
                if not stype:
                    stype = m.get("stationType", "")
        if not prec or not block:
            print(f"  [スキップ] {name or sid}: 地点番号を特定できません")
            continue
        if not stype:
            stype = "気象官署" if len(block) >= 5 and block.startswith("47") else "アメダス"
        out.append(Station(sid, name or sid, prec, block, stype, elems, status))
    return out


# ------------------------------------------------------------
# 期間の分割（連続モードのみ。複数年モードは分割せず1リクエスト）
# ------------------------------------------------------------
def date_chunks(start: date, end: date, unit: str):
    cur = start
    while cur <= end:
        if unit == "month":
            if cur.month == 12:
                last = date(cur.year, 12, 31)
            else:
                last = date(cur.year, cur.month + 1, 1) - timedelta(days=1)
            chunk_end = min(last, end)
            nxt_month = cur.month + 1
            nxt_year = cur.year + (1 if nxt_month > 12 else 0)
            nxt_month = 1 if nxt_month > 12 else nxt_month
            nxt = date(nxt_year, nxt_month, 1)
        else:
            chunk_end = min(date(cur.year, 12, 31), end)
            nxt = date(cur.year + 1, 1, 1)
        yield cur, chunk_end
        cur = nxt


# ------------------------------------------------------------
# 1リクエストのデータ量が気象庁の上限を超えないよう分割する
#   気象庁の判定式（top.2.1.js より）:
#     地点数 × 項目数 × 期間の点数(nOfPr) × 重み ≤ seigen(=44000)
#   本ツールは1リクエスト=1地点なので、項目数 × 点数 × 重み ≤ 上限 になるよう分割する。
# ------------------------------------------------------------
SEIGEN = 44000
VOLUME_LIMIT = 40000  # 44000に対して余裕を持たせた実効上限


def _safe_date(y: int, m: int, d: int) -> date:
    last = calendar.monthrange(y, m)[1]
    return date(y, m, min(d, last))


def _days_between(y1, m1, d1, y2, m2, d2) -> int:
    return abs((_safe_date(y2, m2, d2) - _safe_date(y1, m1, d1)).days) + 1


def count_periods(aggrg_type: int, inter: int, y1, y2, m1, m2, d1, d2) -> int:
    """気象庁 countPrNum を移植。期間の点数(nOfPr)を返す。ymd=[y1,y2,m1,m2,d1,d2]。"""
    if inter == 1:  # 連続した期間
        if aggrg_type in (1, 8, 9):
            diff = _days_between(y1, m1, d1, y2, m2, d2)
            if aggrg_type == 9:
                diff *= 24
        elif aggrg_type in (2, 4):
            sub = 6 if aggrg_type == 2 else 3
            diff = abs((y2 - y1) * 12 * sub + (m2 - m1) * sub + (d2 - d1)) + 1
        elif aggrg_type in (5, 6):
            diff = abs(y2 * 12 + m2 - y1 * 12 - m1) + 1
        else:
            diff = _days_between(y1, m1, d1, y2, m2, d2)
    else:  # 特定の期間を複数年分
        if aggrg_type in (1, 8, 9):
            dt1 = _safe_date(y1, m2, d2)
            dt2 = _safe_date(y1, m1, d1)
            if dt1 < dt2:
                dt1 = _safe_date(y1 + 1, m2, d2)
            diff_day = abs((dt1 - dt2).days) + 1
            diff_year = abs(y2 - y1) + 1
            diff = diff_year * diff_day
            if aggrg_type == 9:
                diff *= 24
        elif aggrg_type in (2, 4):
            sub = 6 if aggrg_type == 2 else 3
            yd = abs(y1 - y2) + 1
            if m1 < m2 or (m1 == m2 and d1 <= d2):
                md = abs((m2 - m1) * sub + (d2 - d1)) + 1
            else:
                md = abs(12 * sub - ((m1 - m2) * sub + (d1 - d2))) + 1
            diff = yd * md
        elif aggrg_type in (5, 6):
            yd = abs(y1 - y2) + 1
            md = (m2 - m1 + 1) if m1 <= m2 else (12 - (m1 - m2) + 1)
            diff = yd * md
        else:
            diff = abs(y2 - y1) + 1
    return int(diff)


def build_plan(inter_type, aggrg_type, n_el, y1, m1, d1, y2, m2, d2, chunk_unit):
    """各リクエストが上限内に収まる計画を作る。戻り値: [(inter, [Y1,Y2,M1,M2,D1,D2], 名前suffix), ...]"""
    weight = 1.5 if aggrg_type == 8 else 1.0

    def cost(inter, Y1, Y2, M1, M2, D1, D2):
        return n_el * weight * count_periods(aggrg_type, inter, Y1, Y2, M1, M2, D1, D2)

    def split_cont(cs, ce):
        # 連続期間を、上限を超えるなら日数で二分して収める
        if cost(1, cs.year, ce.year, cs.month, ce.month, cs.day, ce.day) <= VOLUME_LIMIT or cs >= ce:
            return [("1", [cs.year, ce.year, cs.month, ce.month, cs.day, ce.day], f"{cs.isoformat()}_{ce.isoformat()}")]
        mid = cs + (ce - cs) // 2
        return split_cont(cs, mid) + split_cont(mid + timedelta(days=1), ce)

    specs = []
    if inter_type == "1":
        d1c = min(d1, calendar.monthrange(y1, m1)[1])
        d2c = min(d2, calendar.monthrange(y2, m2)[1])
        for cs, ce in date_chunks(date(y1, m1, d1c), date(y2, m2, d2c), chunk_unit):
            specs.extend(split_cont(cs, ce))
    else:
        d1c = min(d1, calendar.monthrange(2000, m1)[1])
        d2c = min(d2, calendar.monthrange(2000, m2)[1])
        per_year = cost(2, 2000, 2000, m1, m2, d1c, d2c)  # 1年分の量
        if per_year <= VOLUME_LIMIT:
            batch = max(1, int(VOLUME_LIMIT // per_year))
            y = y1
            while y <= y2:
                by2 = min(y + batch - 1, y2)
                suffix = f"{y}-{by2}_各年{m1:02d}{d1c:02d}-{m2:02d}{d2c:02d}"
                specs.append(("2", [y, by2, m1, m2, d1c, d2c], suffix))
                y = by2 + 1
        else:
            # 1年分でも上限超 → 各年を連続期間として月/年分割
            for Y in range(y1, y2 + 1):
                ys = min(d1, calendar.monthrange(Y, m1)[1])
                ye = min(d2, calendar.monthrange(Y, m2)[1])
                for cs, ce in date_chunks(date(Y, m1, ys), date(Y, m2, ye), chunk_unit):
                    specs.extend(split_cont(cs, ce))
    return specs


# ------------------------------------------------------------
# 取得（リトライ＋エラーHTML検出つき）
# ------------------------------------------------------------
def looks_like_csv(content: bytes) -> bool:
    head = content[:200].lstrip()
    if head[:1] == b"<":
        return False
    return len(content) > 0


def fetch_data(session, station_nums, ymd, aggrg_period, elements, inter_annual_type, sleep_sec):
    payload = WeatherDataPayload(
        stationNumList=list(station_nums),
        aggrgPeriod=int(aggrg_period),
        elementNumList=[[code, ""] for code in elements],
        interAnnualType=int(inter_annual_type),
        ymdList=[str(x) for x in ymd],
    )
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = session.post(SHOW_URL, data=payload.to_post_data(),
                                headers={"Referer": ROOT_URL}, timeout=60)
            resp.raise_for_status()
            if not looks_like_csv(resp.content):
                raise RuntimeError("CSV以外の応答（混雑またはデータ量超過の可能性）")
            return resp.content
        except Exception as e:
            last_err = e
            if attempt < MAX_RETRIES:
                time.sleep(sleep_sec * attempt)
    raise last_err


def save_csv(content: bytes, output_path: Path) -> None:
    try:
        output_path.write_text(content.decode("cp932"), encoding="utf-8-sig", newline="")
    except UnicodeDecodeError:
        output_path.write_bytes(content)


# ------------------------------------------------------------
# 地点ごとに複数チャンクを1ファイルへ結合
#   気象庁CSVは先頭に数行のヘッダー（時刻・地点名・項目名など）があり、
#   データ行は「年で始まる行」。先頭ファイルのヘッダー＋全ファイルのデータ行を連結する。
# ------------------------------------------------------------
def merge_station_files(paths: List[Path], out_path: Path) -> None:
    header_lines: Optional[List[str]] = None
    data_all: List[str] = []
    for p in paths:
        lines = p.read_text(encoding="utf-8-sig").splitlines()
        di = next((k for k, ln in enumerate(lines) if ln[:1].isdigit()), len(lines))
        if header_lines is None:
            header_lines = lines[:di]
        data_all.extend(lines[di:])
    out_lines = (header_lines or []) + data_all
    out_path.write_text("\r\n".join(out_lines) + "\r\n", encoding="utf-8-sig", newline="")


# ============================================================
# フォームUI（ipywidgets）
# ============================================================
CODE_TO_CATEGORY = {}
for v, lbl, cat in ELEMENTS_HOURLY:
    CODE_TO_CATEGORY[("9", v)] = cat
for v, lbl, kikan, cat in ELEMENTS_OTHER:
    for k in kikan:
        CODE_TO_CATEGORY[(k, v)] = cat

_this_year = date.today().year
_years = [str(y) for y in range(_this_year, 1899, -1)]
_months = [str(m) for m in range(1, 13)]
_days = [str(d) for d in range(1, 32)]
_yesterday = date.today() - timedelta(days=1)  # 気象庁データは前日ぶんまでが目安

BOX_WIDTH = "560px"
_row = lambda *w: widgets.HBox(list(w))

style = {"description_width": "96px"}
LW = widgets.Layout(width="150px")

paste_area = widgets.Textarea(
    placeholder="観測所ファインダーで「Colab用にコピー」を押し、ここに貼り付け（Ctrl+V）。",
    description="地点リスト", style=style,
    layout=widgets.Layout(width=BOX_WIDTH, height="100px"))
mode_dd = widgets.RadioButtons(
    options=[("連続した期間で表示する", "1"), ("特定の期間を複数年分、表示する", "2")],
    value="1", description="期間の種類", style=style, layout=widgets.Layout(width=BOX_WIDTH))
period_dd = widgets.Dropdown(
    options=[(lbl, code) for code, lbl, _u in PERIOD_OPTIONS],
    value="1", description="データの種類", style=style, layout=widgets.Layout(width="300px"))
elem_sel = widgets.SelectMultiple(
    options=[], description="観測項目", style=style,
    layout=widgets.Layout(width=BOX_WIDTH, height="150px"))

# 既定: 開始 1976/1/1（多くのアメダスの観測開始時期）〜 終了は昨日
syear = widgets.Dropdown(options=_years, value="1976", layout=LW)
smonth = widgets.Dropdown(options=_months, value="1", layout=LW)
sday = widgets.Dropdown(options=_days, value="1", layout=LW)
eyear = widgets.Dropdown(options=_years, value=str(_yesterday.year), layout=LW)
emonth = widgets.Dropdown(options=_months, value=str(_yesterday.month), layout=LW)
eday = widgets.Dropdown(options=_days, value=str(_yesterday.day), layout=LW)

period_desc = widgets.HTML()
start_row = _row(widgets.Label("開始", layout=widgets.Layout(width="40px")), syear, smonth, sday)
end_row = _row(widgets.Label("終了", layout=widgets.Layout(width="40px")), eyear, emonth, eday)

SLEEP_SEC = 3.0  # 気象庁サーバーへの配慮のため固定
merge_mode = widgets.RadioButtons(
    options=[("自動（ファイルが少ない方でまとめる）", "auto"),
             ("地点ごとに1ファイル", "station"),
             ("期間ごとに1ファイル", "period")],
    value="auto", description="まとめ方", style=style, layout=widgets.Layout(width=BOX_WIDTH))
merge_hint = widgets.HTML(
    "<span style='color:#555;font-size:12px'>※ 地点数と期間の分割数を比べ、少ない方でまとめるとファイル数を減らせます。"
    "「自動」は実行時に少ない方を選びます。</span>")
run_btn = widgets.Button(description="ダウンロード開始", button_style="success",
                         layout=widgets.Layout(width="200px"))
# 進捗表示（ボタンのすぐ下に置き、下までスクロールしなくても状況が分かるようにする・③）
progress = widgets.IntProgress(value=0, min=0, max=1, description="進捗",
                               bar_style="info", style=style,
                               layout=widgets.Layout(width=BOX_WIDTH, visibility="hidden"))
status_label = widgets.HTML(value="")
out = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", padding="6px",
                                            max_height="360px", overflow="auto"))


def refresh_elements(*_):
    pc = period_dd.value
    if pc == "9":
        opts = [(lbl, v) for v, lbl, cat in ELEMENTS_HOURLY]
    else:
        opts = [(lbl, v) for v, lbl, kikan, cat in ELEMENTS_OTHER if pc in kikan]
    elem_sel.options = opts
    # 既定で気温・降水を選択
    default = [v for (lbl, v) in opts if ("気温" in lbl or "降水" in lbl)][:2]
    elem_sel.value = tuple(default) if default else (opts[0][1],) if opts else tuple()


def refresh_desc(*_):
    if mode_dd.value == "1":
        period_desc.value = ("<span style='color:#555'>▼ <b>連続した期間</b>：開始日から終了日まで通しで取得します。</span>")
        start_row.children[0].value = "開始"
        end_row.children[0].value = "終了"
    else:
        period_desc.value = ("<span style='color:#555'>▼ <b>特定の期間を複数年分</b>：各年の「開始（月・日）〜終了（月・日）」を、"
                             "開始年〜終了年の各年について取得します（年の値が年範囲、月日が毎年の対象期間）。</span>")
        start_row.children[0].value = "開始"
        end_row.children[0].value = "終了"


period_dd.observe(refresh_elements, names="value")
mode_dd.observe(refresh_desc, names="value")
refresh_elements()
refresh_desc()


def set_status(html):
    status_label.value = f"<span style='font-size:13px'>{html}</span>"


def on_run(_btn):
    run_btn.disabled = True
    progress.value = 0
    progress.bar_style = "info"
    progress.layout.visibility = "hidden"
    set_status("⏳ 準備中…（設定を確認しています）")
    out.clear_output()
    completed = False
    with out:
        try:
            csv_text = paste_area.value.strip()
            if not csv_text:
                print("地点リストを貼り付け欄に貼り付けてください（ファインダーの「Colab用にコピー」）。")
                return
            stations = read_stations_from_text(csv_text)
            if not stations:
                print("有効な地点がCSVから読み取れませんでした。")
                return

            inter_type = mode_dd.value
            period_code = period_dd.value
            period_label = dict((c, l) for c, l, _u in PERIOD_OPTIONS)[period_code]
            chunk_unit = dict((c, u) for c, _l, u in PERIOD_OPTIONS)[period_code]
            element_codes = list(elem_sel.value)
            if not element_codes:
                print("観測項目を1つ以上選択してください。")
                return
            sel_cats = {CODE_TO_CATEGORY.get((period_code, c), "") for c in element_codes}
            sel_cats.discard("")
            sleep_sec = SLEEP_SEC

            y1, m1, d1 = int(syear.value), int(smonth.value), int(sday.value)
            y2, m2, d2 = int(eyear.value), int(emonth.value), int(eday.value)
            aggrg_type = int(period_code)

            # 期間の妥当性チェック
            if inter_type == "2":
                if y1 > y2:
                    print("開始年は終了年以前にしてください。")
                    return
            else:
                if date(y1, m1, min(d1, calendar.monthrange(y1, m1)[1])) > date(y2, m2, min(d2, calendar.monthrange(y2, m2)[1])):
                    print("開始日は終了日以前にしてください。")
                    return

            # リクエスト計画（期間軸: 各リクエストが1地点で上限内に収まるよう分割）
            plan = build_plan(inter_type, aggrg_type, len(element_codes), y1, m1, d1, y2, m2, d2, chunk_unit)
            n_per_station = len(plan)
            n_el = len(element_codes)

            raw_dir = Path("jma_raw")
            raw_dir.mkdir(exist_ok=True)
            merged_dir = Path("jma_data")
            merged_dir.mkdir(exist_ok=True)

            # 1回のリクエストに何地点まとめられるか（期間ごとモード用）。地点数×項目数×点数 ≤ 上限。
            def stations_per_request(inter, ymd):
                cost = max(1, n_el * count_periods(aggrg_type, int(inter), *ymd))
                return max(1, VOLUME_LIMIT // cost)

            # ⑧ まとめ方を決める（自動は「速い＝リクエストが少ない方」。同数ならファイルが少ない方）
            def period_request_count():
                total = 0
                for inter, ymd, _sfx in plan:
                    spr = stations_per_request(inter, ymd)
                    total += -(-len(stations) // spr)  # ceil
                return total

            station_requests = len(stations) * n_per_station
            period_requests = period_request_count()
            mode = merge_mode.value
            if mode == "auto":
                # リクエストが少ない方が速い。同数なら地点ごと（ファイルが少ない）を選ぶ。
                mode = "period" if period_requests < station_requests else "station"

            # 実際のリクエスト一覧を作る
            # req = {nums, inter, ymd, path, display, station(=地点ごと結合用/なければNone)}
            reqs = []
            if mode == "station":
                for s in stations:
                    for inter, ymd, suffix in plan:
                        reqs.append({"nums": [s.station_num()], "inter": inter, "ymd": ymd,
                                     "path": raw_dir / f"{s.name}_{period_label}_{suffix}.csv",
                                     "display": s.name, "station": s.name})
            else:  # period: 上限内で複数地点をまとめて1リクエスト
                for inter, ymd, suffix in plan:
                    spr = stations_per_request(inter, ymd)
                    batches = [stations[i:i + spr] for i in range(0, len(stations), spr)]
                    for bi, batch in enumerate(batches):
                        sfx = suffix if len(batches) == 1 else f"{suffix}_p{bi + 1}"
                        reqs.append({"nums": [b.station_num() for b in batch], "inter": inter, "ymd": ymd,
                                     "path": merged_dir / f"{period_label}_{sfx}.csv",
                                     "display": f"{suffix}（{len(batch)}地点）", "station": None})

            n_requests = len(reqs)
            est_min = n_requests * sleep_sec / 60.0
            discontinued = [s for s in stations if s.status and s.status != "現役"]
            mismatch = [s.name for s in stations if s.elements and sel_cats and not (sel_cats & s.elements)]

            print("── 設定内容 ──")
            print(f"  期間の種類   : {'特定の期間を複数年分' if inter_type == '2' else '連続した期間'}")
            print(f"  データの種類 : {period_label}")
            print(f"  観測項目     : {', '.join(element_codes)}")
            if inter_type == "2":
                print(f"  期間         : 各年 {m1}/{d1} 〜 {m2}/{d2} を {y1}年〜{y2}年")
            else:
                print(f"  期間         : {y1}/{m1}/{d1} 〜 {y2}/{m2}/{d2}（{'1か月' if chunk_unit=='month' else '1年'}ごと）")
            print(f"  対象地点数   : {len(stations)} 地点")
            print(f"  まとめ方     : {'地点ごと' if mode == 'station' else '期間ごと'}"
                  f"{'（自動選択）' if merge_mode.value == 'auto' else ''}")
            print(f"    参考: 地点ごと={station_requests}回 / 期間ごと={period_requests}回（少ない方が速い）")
            print(f"  リクエスト数 : 約 {n_requests} 回（間隔 {sleep_sec}秒 → 推定 約 {est_min:.1f} 分）")
            if n_per_station > 1:
                print(f"    ※ 気象庁の1回あたりデータ量上限を超えないよう自動分割しています")
            if discontinued:
                print(f"  ⚠ 廃止済み地点が {len(discontinued)} 件（期間により空データの場合あり）")
            if mismatch:
                ex = "、".join(mismatch[:5]) + ("…" if len(mismatch) > 5 else "")
                print(f"  ⚠ 選んだ項目を観測していない可能性のある地点 {len(mismatch)} 件（例: {ex}）")
            print()

            progress.max = max(1, n_requests)
            progress.layout.visibility = "visible"
            set_status("🌐 気象庁へ接続中…")
            session = requests.Session()
            session.headers.update({"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"})
            session.get(ROOT_URL, timeout=30)
            time.sleep(sleep_sec)

            n_ok = n_skip = n_fail = 0
            done = 0
            failures: List[str] = []
            start_time = time.time()

            def update_status(current_name):
                elapsed = time.time() - start_time
                eta = ""
                if done > 0:
                    remain = (elapsed / done) * (n_requests - done)
                    eta = f"・残り 約 {remain/60:.1f} 分"
                set_status(
                    f"📥 取得中: <b>{current_name}</b>　{done}/{n_requests} 件"
                    f"（成功 {n_ok}／失敗 {n_fail}）{eta}"
                )

            saved_by_station: Dict[str, List[Path]] = {}  # 地点ごとモードの結合用
            for req in reqs:
                out_path = req["path"]
                fname = out_path.name
                if out_path.exists() and out_path.stat().st_size > 0:
                    print(f"  スキップ(取得済): {fname}")
                    if req["station"]:
                        saved_by_station.setdefault(req["station"], []).append(out_path)
                    n_skip += 1
                    done += 1
                    progress.value = done
                    update_status(req["display"])
                    continue
                print(f"取得中: {req['display']}")
                update_status(req["display"])
                try:
                    content = fetch_data(session, req["nums"], req["ymd"], period_code,
                                         element_codes, req["inter"], sleep_sec)
                    save_csv(content, out_path)
                    if req["station"]:
                        saved_by_station.setdefault(req["station"], []).append(out_path)
                    print(f"  保存: {fname}")
                    n_ok += 1
                except Exception as e:
                    print(f"  [エラー] {fname}: {e}")
                    failures.append(f"{fname}: {e}")
                    n_fail += 1
                done += 1
                progress.value = done
                update_status(req["display"])
                time.sleep(sleep_sec)

            # 地点ごとモードは、地点ごとに時期チャンクを1ファイルへ結合する
            # （期間ごとモードは取得した応答がそのまま複数地点まとめ済みなので結合不要）
            if mode == "station" and saved_by_station:
                set_status("🗂 地点ごとにまとめています…")
                for name, paths in saved_by_station.items():
                    try:
                        merge_station_files(paths, merged_dir / f"{name}_{period_label}.csv")
                    except Exception as e:
                        print(f"  [結合エラー] {name}: {e}")

            progress.bar_style = "success" if n_fail == 0 else "warning"
            set_status(f"✅ 完了：成功 {n_ok}／スキップ {n_skip}／失敗 {n_fail}　（下の枠に詳細）")
            print()
            print("── 結果サマリ ──")
            print(f"  成功: {n_ok} / スキップ(取得済): {n_skip} / 失敗: {n_fail}")
            if failures:
                print("  失敗した項目:")
                for f in failures:
                    print(f"    - {f}")

            has_files = any(merged_dir.iterdir())
            if has_files:
                print()
                print("ZIPにまとめてダウンロードします...")
                shutil.make_archive("jma_data_result", "zip", merged_dir)
                files.download("jma_data_result.zip")
            else:
                print("保存できたファイルがないため、ZIPは作成しませんでした。")
            completed = True
        finally:
            run_btn.disabled = False
            if not completed:
                progress.layout.visibility = "hidden"
                set_status("⚠️ 中断しました。下の枠のメッセージをご確認ください。")


run_btn.on_click(on_run)

def _section(title):
    return widgets.HTML(f"<div style='margin:10px 0 2px;font-weight:700;color:#1C7C8C;"
                        f"border-bottom:2px solid #E4F0F1;padding-bottom:2px'>{title}</div>")

form = widgets.VBox([
    widgets.HTML("<h3 style='margin:4px 0'>気象庁データ 一括ダウンロード</h3>"
                 "<div style='color:#555;font-size:13px'>下の 1 → 2 → 3 の順に選び、「ダウンロード開始」を押してください。</div>"),

    _section("1. 地点リストを貼り付け"),
    paste_area,

    _section("2. データの種類 ＆ 観測項目"),
    period_dd,
    elem_sel,

    _section("3. 期間の種類 ＆ 開始〜終了"),
    mode_dd,
    period_desc,
    start_row,
    end_row,

    _section("まとめ方"),
    merge_mode,
    merge_hint,

    run_btn,
    progress,
    status_label,
    out,
], layout=widgets.Layout(max_width="640px"))

display(form)
